# Lesson 2: Diffusion samplers


## Hyperparameters

All experiment settings are collected below. Edit this panel, then **Run All**.


In [ ]:
# HYPERPARAMETERS — edit here, then Run All

# Objective and target distribution
TEMPERATURE = 1.0             # Final target temperature T (must be positive).
INITIAL_TEMPERATURE = 2.0     # Starting temperature for the entropy curriculum.
TEMPERATURE_ANNEAL_STEPS = 700  # Updates for the linear change to TEMPERATURE.
NUM_MODES = 40                # Number of equally weighted Gaussian target modes.
MODE_BOUND = 40.0             # Mode centers lie in [-MODE_BOUND, MODE_BOUND]^2.
TARGET_VARIANCE = 1.0         # Variance of each target component per coordinate.
TARGET_SEED = 0               # Seed that fixes the target's component locations.

# Diffusion family and score network
NUM_DIFFUSION_STEPS = 24      # K transitions, with K+1 noise coefficients.
PRIOR_STD = 30.0              # Fixed Gaussian prior scale nu; not learned by the code.
BETA_START = 5e-4             # Fixed first coefficient beta_0.
BETA_END = 2e-3               # Fixed last coefficient beta_K.
DIFFUSION_STEP_SIZE = 1.0     # Delta; each noise coefficient is delta_k = beta_k * Delta.
LEARN_DIFFUSION_SCHEDULE = False  # Optimize the monotone interior beta coefficients.
SCORE_WIDTH = 96             # Score-network hidden width; gradient gate uses half.
# The comparison always runs both estimators, with Langevin off and on.

# Optimization (shared by all four runs)
SEED = 11                    # Shared seed for network initialization and training.
TRAINING_STEPS = 800         # Number of optimizer updates per configuration.
BATCH_SIZE = 256             # Sampled diffusion paths per update (at least 2).
LEARNING_RATE = 5e-4          # Adam learning rate for the score network.
SCHEDULE_LEARNING_RATE = 3e-4 # Adam learning rate for learned schedule parameters.
MAX_GRAD_NORM = 10.0         # Upper bound on the norm of an optimizer gradient.

# Evaluation and animation
LOG_EVERY = 40               # Record diagnostics every this many updates.
NUM_EVAL_SAMPLES = 5000      # Fresh samples for the final mode-recovery diagnostic.
EVALUATION_SEED = 31         # Seed for the mode-recovery samples.
MODE_CAPTURE_RADIUS = 1.25   # Maximum distance to a center to count captured mass.
MINIMUM_MODE_FRACTION = 0.005 # Minimum captured mass to mark a mode as recovered.
NUM_GIF_SAMPLES = 600        # Fixed-noise particles per panel and path-loss estimate.
ANIMATION_SEED = 10011       # Shared seed for the fixed prior and transition noise.
GIF_FPS = 1.5                # Saved animation frames per second.
GIF_DPI = 92                 # Saved animation resolution in dots per inch.
PLOT_BOUND = 56.0            # Plot both coordinates over [-PLOT_BOUND, PLOT_BOUND].
GRID_RESOLUTION = 360        # Grid points per coordinate for the target contours.
CONTOUR_LEVELS = 80          # Number of target-reward contour levels.
REWARD_PLOT_RANGE = 100.0    # Display rewards within this distance below the maximum.

# Numerical stability (implementation settings)
LANGEVIN_GRADIENT_CLIP = 1e2 # Bound on each analytic target-gradient coordinate.
LANGEVIN_DRIFT_CLIP = 1e4    # Bound on each preconditioned drift coordinate.


## Discrete diffusion setup

A diffusion sampler replaces the single Gaussian from Lesson 1 by a chain of Gaussian transitions. The learned reverse process maps the broad prior $q_K=\mathcal N(0,\nu^2 I)$, with $\nu=30$, toward the seed-0 GMM-40 target.

We use the same discrete-time updates as the slides. The noise coefficients are $\delta_j=\beta_j\Delta_j$, with $\Delta_j=1$ in this exercise. There are $K=24$ transitions and $K+1$ coefficients indexed $j=0,\ldots,K$.

For the pair $(x^{k-1},x^k)$, the forward kernel uses $\delta_{k-1}$ and the reverse kernel uses $\delta_k$. The shared training code passes the appropriate coefficient as the `delta` argument. `step_index` is the mathematical $k\in\{1,\ldots,K\}$.

Your task is to implement both discrete steps and their kernel log probabilities. The reparameterization and log-derivative estimators are already provided.


## Setup and experiment flags

Use the hyperparameter panel above for every experiment setting. Run this setup again after changing it, then run the remaining cells.

`LEARN_DIFFUSION_SCHEDULE=True` learns the monotone interior coefficients while keeping both endpoints fixed. The Gaussian prior scale `PRIOR_STD` is configurable but is **not learned**; learning prior parameters would require extending the code.

The four runs compare reparameterization and log derivative, each with and without Langevin preconditioning. The latter parameterizes the score as

$$u_\Psi(x,k)=-x/\nu^2+\mathrm{NN}_1(k,x)+\mathrm{NN}_2(k)\odot\nabla_x\log p_T(x).$$

The schedule stores $\beta_0,\ldots,\beta_K$. Its $K-1$ interior coefficients can be trained; forward and reverse transitions on the same pair use adjacent coefficients.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.distributions import Independent, Normal

lesson_directory = Path("L2-DiffusionSamplers")
if not lesson_directory.is_dir():
    lesson_directory = Path(".")
sys.path.insert(0, str(lesson_directory.resolve()))

from lesson2_common import configure_target

configure_target(
    num_modes=NUM_MODES, mode_bound=MODE_BOUND,
    target_variance=TARGET_VARIANCE, seed=TARGET_SEED,
    plot_bound=PLOT_BOUND, grid_resolution=GRID_RESOLUTION,
    contour_levels=CONTOUR_LEVELS, reward_plot_range=REWARD_PLOT_RANGE,
)

from lesson2_common import (
    DiffusionKernels,
    check_kernel_functions,
    evaluate_mode_recovery,
    plot_reward_landscape,
    plot_training_summary,
    save_training_comparison_animation,
    train_sampler,
)

torch.manual_seed(SEED)
EXPERIMENTS = [
    (estimator, use_langevin)
    for estimator in ("reparameterization", "log_derivative")
    for use_langevin in (False, True)
]
print(f"VP transitions: {NUM_DIFFUSION_STEPS}; coefficients: {NUM_DIFFUSION_STEPS + 1}")
print(f"delta = beta * Delta, Delta={DIFFUSION_STEP_SIZE:g}; beta: {BETA_START:.4f} -> {BETA_END:.4f}")
print(f"Prior standard deviation: {PRIOR_STD:g}; experiments: {EXPERIMENTS}")

## Target and prior

The target is the equal-weight GMM-40 benchmark: PyTorch seed 0, 40 means sampled uniformly from $[-40,40]^2$, and component covariance $I$. The terminal prior is $q_K=\mathcal N(0,30^2I)$.

In [ ]:
fig, axis = plt.subplots(figsize=(6.4, 5.2))
plot_reward_landscape(axis)
axis.legend(loc="lower right", fontsize=8)
axis.set_title(f"Lesson 2 target: seed-{TARGET_SEED} GMM-{NUM_MODES}")
plt.tight_layout()
plt.show()

## Task 1: implement the discrete forward step

The forward step shrinks the previous state and adds independent Gaussian noise:

$$x^k=(1-\tfrac12\delta_{k-1})x^{k-1}+\nu\sqrt{\delta_{k-1}}\,\varepsilon_{k-1},\qquad\varepsilon_{k-1}\sim\mathcal N(0,I).$$

Thus $p(x^k\mid x^{k-1})=\mathcal N((1-\tfrac12\delta_{k-1})x^{k-1},\nu^2\delta_{k-1}I)$, with $\nu=\texttt{PRIOR\_STD}=30$.

**Task:** implement this Gaussian transition. Support the supplied fixed noise and the `reparameterize` flag, returning a state with the same shape as `previous_state`.

**Optional hints** — click a label to expand.

<details>
<summary><strong>Get a hint: Gaussian parameters</strong></summary>
<p>The mean is <code>(1 - 0.5 * delta) * previous_state</code> and the standard deviation is <code>PRIOR_STD * torch.sqrt(delta)</code>. Use the imported <code>Independent(Normal(mean, std), 1)</code> to treat the two state coordinates as one event. <code>Normal</code> expects a standard deviation, not a variance.</p>
</details>

<details>
<summary><strong>Get a hint: fixed noise</strong></summary>
<p>When <code>noise is not None</code>, compute <code>sample = mean + std * noise</code> and return <code>sample</code> when <code>reparameterize</code> is true, otherwise <code>sample.detach()</code> using the supplied standard-normal noise, which has the same shape as the state. The training animation supplies fixed noise to follow the same particles across iterations.</p>
</details>

<details>
<summary><strong>Get a hint: sampling and shapes</strong></summary>
<p>Without fixed noise, use <code>distribution.rsample()</code> if <code>reparameterize</code> is true, and <code>distribution.sample()</code> otherwise. The mean already has shape <code>(batch_size, 2)</code>, so call these methods without an extra sample shape. Both return <code>(batch_size, 2)</code>; <code>rsample()</code> keeps the gradient path through the Gaussian parameters.</p>
</details>

In [ ]:
def forward_sde_step(
    previous_state: torch.Tensor,
    delta: torch.Tensor,
    *,
    reparameterize: bool,
    noise: torch.Tensor | None = None,
) -> torch.Tensor:
    """One variance-preserving forward noising step."""
    # TODO: Implement the forward discrete VP step. Optional hints are above.
    raise NotImplementedError("Implement the forward discrete VP step")

## Task 2: implement the learned reverse step

The optimal score $s_k^*(x)=\nabla_x\log p_k(x)$ is intractable in general. The score network $u_\Psi(x,k)$ approximates it and gives the reverse mean

$$\mu_\theta(x^k,k)=(1+\tfrac12\delta_k)x^k+\nu^2\delta_k u_\Psi(x^k,k).$$

Sample $x^{k-1}=\mu_\theta(x^k,k)+\nu\sqrt{\delta_k}\,\varepsilon_k$, with $\varepsilon_k\sim\mathcal N(0,I)$. Equivalently,

$$q_\theta(x^{k-1}\mid x^k)=\mathcal N(\mu_\theta(x^k,k),\nu^2\delta_k I).$$

The network starts at the prior score $u_\Psi(x,k)=-x/\nu^2$. These small Euler steps preserve the prior approximately, rather than exactly.

**Task:** implement this reverse transition using the supplied score network. Support fixed noise and the `reparameterize` flag, returning a state with the same shape as `noisy_state`.

**Optional hints** — click a label to expand.

<details>
<summary><strong>Get a hint: calling the score network</strong></summary>
<p>Call <code>score_network(noisy_state, step_index, num_steps)</code>. Pass the supplied step arguments directly: <code>step_index</code> is k from 1 through K; the network handles time encoding. The returned score has the same shape as <code>noisy_state</code>, usually <code>(batch_size, 2)</code>.</p>
</details>

<details>
<summary><strong>Get a hint: the reverse Gaussian</strong></summary>
<p>Use <code>(1 + 0.5 * delta) * noisy_state + delta * PRIOR_STD**2 * score</code> for the mean and <code>PRIOR_STD * torch.sqrt(delta)</code> for the standard deviation. Construct <code>Independent(Normal(mean, std), 1)</code>. The variance factor <code>PRIOR_STD**2</code> in the mean is required by the scaled process in this lesson.</p>
</details>

<details>
<summary><strong>Get a hint: fixed noise and sampling</strong></summary>
<p>If <code>noise is not None</code>, compute <code>sample = mean + std * noise</code> and return <code>sample</code> when <code>reparameterize</code> is true, otherwise <code>sample.detach()</code>. Otherwise choose <code>distribution.rsample()</code> when <code>reparameterize</code> is true, or <code>distribution.sample()</code> when it is false. The distribution is already batched, so no extra sample shape is needed. Keep the score and Gaussian parameters attached so reparameterized samples can carry gradients into the network.</p>
</details>

In [ ]:
def reverse_sde_step(
    score_network,
    noisy_state: torch.Tensor,
    step_index: int,
    delta: torch.Tensor,
    num_steps: int,
    *,
    reparameterize: bool,
    noise: torch.Tensor | None = None,
) -> torch.Tensor:
    """One learned reverse denoising step."""
    # TODO: Implement the reverse discrete VP step. Optional hints are above.
    raise NotImplementedError("Implement the reverse discrete VP step")

## Task 3: implement the forward-kernel log probability

Evaluate $\log p(x^k\mid x^{k-1})$ under the same Gaussian used by the forward step.

**Optional hints** — click a label to expand.

<details>
<summary><strong>Get a hint: the conditional distribution</strong></summary>
<p>Reconstruct the forward Gaussian conditioned on <code>previous_state</code>: mean <code>(1 - 0.5 * delta) * previous_state</code> and standard deviation <code>PRIOR_STD * torch.sqrt(delta)</code>. Wrap it as <code>Independent(Normal(mean, std), 1)</code>. These parameters must match <code>forward_sde_step</code>.</p>
</details>

<details>
<summary><strong>Get a hint: log probabilities and shapes</strong></summary>
<p>Evaluate the supplied next state with <code>distribution.log_prob(noisy_state)</code>. For input states of shape <code>(batch_size, 2)</code>, this returns <code>(batch_size,)</code>. <code>Independent</code> already sums over the two coordinates; return one log probability per batch element without averaging across the batch. This function evaluates an existing transition, so no sampling is needed.</p>
</details>

<details>
<summary><strong>Get a hint: gradients</strong></summary>
<p>Use tensor operations such as <code>torch.sqrt(delta)</code> and keep the Gaussian parameters and returned log probabilities attached to the computation graph. The learned diffusion schedule needs gradients through <code>delta</code>.</p>
</details>

In [ ]:
def forward_kernel_log_prob(
    previous_state: torch.Tensor,
    noisy_state: torch.Tensor,
    delta: torch.Tensor,
) -> torch.Tensor:
    """Evaluate log p(x^k | x^{k-1})."""
    # TODO: Implement the forward-kernel log probability. Optional hints are above.
    raise NotImplementedError("Implement the forward-kernel log probability")

## Task 4: implement the reverse-kernel log probability

Evaluate $\log q_\theta(x^{k-1}\mid x^k)$ using exactly the same score-conditioned mean as the reverse step.

**Optional hints** — click a label to expand.

<details>
<summary><strong>Get a hint: the score-conditioned distribution</strong></summary>
<p>Call <code>score_network(noisy_state, step_index, num_steps)</code>. Reconstruct the same reverse mean <code>(1 + 0.5 * delta) * noisy_state + delta * PRIOR_STD**2 * score</code> and standard deviation <code>PRIOR_STD * torch.sqrt(delta)</code> as in <code>reverse_sde_step</code>. Construct <code>Independent(Normal(mean, std), 1)</code>.</p>
</details>

<details>
<summary><strong>Get a hint: which state to evaluate</strong></summary>
<p>The reverse kernel is conditioned on <code>noisy_state</code> and assigns a density to <code>previous_state</code>. Evaluate <code>distribution.log_prob(previous_state)</code>. For states of shape <code>(batch_size, 2)</code>, return the resulting <code>(batch_size,)</code> values. The two coordinates are already summed by <code>Independent</code>; no additional sampling or batch reduction is needed.</p>
</details>

<details>
<summary><strong>Get a hint: gradients</strong></summary>
<p>Keep the score, Gaussian parameters, and log probabilities attached. Gradients through this log probability train the score network and, when enabled, the diffusion schedule through <code>delta</code>. Use tensor operations such as <code>torch.sqrt(delta)</code> to preserve those paths.</p>
</details>

In [ ]:
def reverse_kernel_log_prob(
    score_network,
    previous_state: torch.Tensor,
    noisy_state: torch.Tensor,
    step_index: int,
    delta: torch.Tensor,
    num_steps: int,
) -> torch.Tensor:
    """Evaluate log q_theta(x^{k-1} | x^k)."""
    # TODO: Implement the reverse-kernel log probability. Optional hints are above.
    raise NotImplementedError("Implement the reverse-kernel log probability")

## Check the implementations

These checks verify shapes, finite values, and gradients through the score network and schedule.

In [ ]:
kernels = DiffusionKernels(
    forward_sde_step=forward_sde_step,
    reverse_sde_step=reverse_sde_step,
    forward_kernel_log_prob=forward_kernel_log_prob,
    reverse_kernel_log_prob=reverse_kernel_log_prob,
)
check_kernel_functions(kernels, prior_std=PRIOR_STD, num_steps=NUM_DIFFUSION_STEPS)

## Path-space objective: already implemented

Marginalizing a diffusion path is a data-processing operation, so

$$\mathrm{KL}(q_\theta(x^0)\|p_T(x^0))\leq\mathrm{KL}(q_\theta(x^{0:K})\|p_T(x^{0:K})).$$

The shared code expands this tractable joint KL into terminal negative reward plus forward/reverse log-kernel ratios. Both the pathwise and leave-one-out score estimators from Lesson 1 are already complete.

## Train the four sampler configurations

In [ ]:
comparison = []
trained_samplers = {}
for gradient_estimator, use_langevin in EXPERIMENTS:
    sampler, history = train_sampler(
        kernels,
        gradient_estimator=gradient_estimator,
        learn_schedule=LEARN_DIFFUSION_SCHEDULE,
        use_langevin_preconditioning=use_langevin,
        seed=SEED,
        steps=TRAINING_STEPS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        schedule_learning_rate=SCHEDULE_LEARNING_RATE,
        max_grad_norm=MAX_GRAD_NORM,
        temperature=TEMPERATURE,
        initial_temperature=INITIAL_TEMPERATURE,
        temperature_anneal_steps=TEMPERATURE_ANNEAL_STEPS,
        num_steps=NUM_DIFFUSION_STEPS,
        prior_std=PRIOR_STD,
        beta_start=BETA_START,
        beta_end=BETA_END,
        diffusion_step_size=DIFFUSION_STEP_SIZE,
        score_width=SCORE_WIDTH,
        langevin_gradient_clip=LANGEVIN_GRADIENT_CLIP,
        langevin_drift_clip=LANGEVIN_DRIFT_CLIP,
        animation_seed=ANIMATION_SEED,
        log_every=LOG_EVERY,
        num_animation_samples=NUM_GIF_SAMPLES,
    )
    comparison.append((gradient_estimator, use_langevin, history))
    trained_samplers[(gradient_estimator, use_langevin)] = sampler
    _, recovered_modes = evaluate_mode_recovery(
        sampler, num_samples=NUM_EVAL_SAMPLES, seed=EVALUATION_SEED,
        capture_radius=MODE_CAPTURE_RADIUS, minimum_fraction=MINIMUM_MODE_FRACTION,
    )
    label = f"{gradient_estimator:18s} | Langevin={str(use_langevin):5s}"
    print(
        f"{label} | reward {history['mean_reward'][0]:7.2f} -> "
        f"{history['mean_reward'][-1]:7.2f} | recovered {recovered_modes.sum().item():2d}/{NUM_MODES}"
    )

reference_sampler = trained_samplers[("reparameterization", True)]
reference_history = next(
    history for estimator, langevin, history in comparison
    if estimator == "reparameterization" and langevin
)
plot_training_summary(reference_sampler, reference_history)
plt.show()

## Animate samples over training

Fixed prior and transition noise make each dot move smoothly between frames and make the four panels directly comparable. The GIF shows reparameterization and log derivative, each with and without Langevin preconditioning. The two loss panels use the same fixed-noise path objective for a fair comparison; their dashed vertical lines and point markers track the current frame.

In [ ]:
output_directory = Path("L2-DiffusionSamplers")
if not output_directory.is_dir():
    output_directory = Path(".")
gif_path = output_directory / "diffusion_sampler_training.gif"
save_training_comparison_animation(comparison, gif_path, fps=GIF_FPS, dpi=GIF_DPI)
print(f"Saved animation to {gif_path.resolve()}")

from IPython.display import Image, display
display(Image(filename=str(gif_path)))